# MISO/Q1 shard runner (Kaggle GPU, 12h session)

Chạy tuần tự một danh sách shard `algo:seed`, mỗi shard là một job `--train-only`
độc lập. Tự dừng trước khi hết quỹ thời gian; shard chưa chạy được in ra để dán
sang session kế tiếp. Output tải về: MỖI SHARD MỘT FILE ZIP NHỎ trong
`/kaggle/working` (best.pt + manifest + config + history/log + obs_norm —
KHÔNG có latest.pt, KHÔNG có replay buffer).

**Chuẩn bị (1 lần):** upload file `star_ris_miso_src_a8f636f.zip` làm Kaggle
Dataset tên `star-ris-miso-src` rồi Add Input dataset đó vào notebook này.
Settings: Accelerator = GPU T4/P100, Persistence = No, Environment = Pin to
original, Internet = On (để pip cài gymnasium).

In [ ]:
# ==== THAM SỐ - SỬA Ô NÀY CHO TỪNG SESSION ====
RUN_ID = "miso_q1"          # GIỮ NGUYÊN cho mọi session của cùng một đợt thí nghiệm
SHARDS = "maddpg:1000 maddpg:2000 maddpg:3000 maddpg:4000 td3:1000 td3:2000 td3:3000"
TIME_BUDGET_HOURS = 11.3    # dừng an toàn trước mốc 12h của Kaggle
SAFETY_FACTOR = 1.35        # chỉ chạy shard kế nếu còn >= thời_gian_shard_trước * hệ_số
SRC_ZIP = "/kaggle/input/star-ris-miso-src/star_ris_miso_src_a8f636f.zip"
FROZEN_SOURCE_SHA = "2cb81296b206e92483347ede709ad4e53e5fa72182e05842d57083b7a0ac668c"

In [ ]:
# ==== SETUP: giải nén source đã freeze + verify SHA ====
import os, subprocess, sys, time, json, shutil, zipfile, hashlib
T0 = time.time()
WORK = "/kaggle/tmp/miso"
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(SRC_ZIP) as z:
    z.extractall(WORK)
SRC = WORK  # zip chứa source ở gốc
assert os.path.exists(os.path.join(SRC, "main.py")), os.listdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium", "pyyaml", "tqdm"], check=True)
sys.path.insert(0, SRC)
from main import source_tree_sha256
actual = source_tree_sha256(SRC)
assert actual == FROZEN_SOURCE_SHA, f"SOURCE KHONG KHOP FREEZE: {actual}"
print("Source OK, sha =", actual[:16], "| GPU:",
      subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "none")

In [ ]:
# ==== VÒNG LẶP SHARD với time-budget guard ====
OUT = os.path.join(WORK, "out")
os.makedirs(OUT, exist_ok=True)
shard_list = [s for s in SHARDS.split() if s.strip()]
done, skipped, last_dur = [], [], None
for spec in shard_list:
    algo, seed = spec.split(":")
    elapsed_h = (time.time() - T0) / 3600.0
    remain_h = TIME_BUDGET_HOURS - elapsed_h
    need_h = (last_dur or 0.0) * SAFETY_FACTOR
    if last_dur is not None and remain_h < need_h:
        skipped = shard_list[shard_list.index(spec):]
        print(f"DỪNG: còn {remain_h:.2f}h < cần ~{need_h:.2f}h")
        break
    print(f"==== SHARD {spec} (còn {remain_h:.2f}h) ====", flush=True)
    t1 = time.time()
    r = subprocess.run(
        [sys.executable, "main.py", "--config", "config/config.yaml",
         "--train-only", "--algos", algo, "--seeds", seed,
         "--run-id", RUN_ID, "--drop-replay-after-train", "--out", OUT],
        cwd=SRC)
    last_dur = (time.time() - t1) / 3600.0
    shard_dir = os.path.join(OUT, "results_revised", "shards", RUN_ID, f"{algo}_seed{seed}")
    manifest = os.path.join(shard_dir, "shard_manifest.json")
    status = json.load(open(manifest)).get("status") if os.path.exists(manifest) else "missing"
    print(f"    -> {last_dur:.2f}h, status={status}")
    if r.returncode != 0 or status != "completed":
        print(f"    !! SHARD LỖI - giữ log, không đóng gói"); skipped.append(spec); continue
    # Đóng gói TỐI THIỂU: bỏ latest.pt (replay đã bị drop). Aggregation chỉ
    # verify best.pt + effective_config + history/log (đã kiểm chứng end-to-end).
    for root, _, files in os.walk(shard_dir):
        for f in files:
            if f == "latest.pt":
                os.remove(os.path.join(root, f))
    zpath = f"/kaggle/working/{RUN_ID}__{algo}_seed{seed}.zip"
    base = os.path.join(OUT, "results_revised", "shards", RUN_ID)
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(shard_dir):
            for f in files:
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, base))
    done.append(spec)
    print(f"    -> {zpath}  {os.path.getsize(zpath)/1e6:.1f} MB")
print("\n===== TỔNG KẾT =====")
print("Hoàn thành:", " ".join(done) or "(none)")
print("CHƯA CHẠY (dán vào SHARDS của session sau):", " ".join(skipped) or "(none)")

In [ ]:
# ==== LIỆT KÊ FILE TẢI VỀ ====
import glob
total = 0
for p in sorted(glob.glob("/kaggle/working/*.zip")):
    s = os.path.getsize(p); total += s
    print(f"{s/1e6:8.1f} MB  {os.path.basename(p)}")
print(f"{total/1e6:8.1f} MB  TỔNG — chỉ tải các zip này, bỏ qua mọi thứ khác")